## Animation for Science day 

In [1]:
import xarray as xr
from pathlib import Path
# from xhistogram.xarray import histogram as xhist

In [2]:
from dask.distributed import Client
client = Client(n_workers=2, threads_per_worker=3, memory_limit=15e9)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 6,Total memory: 27.94 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37097,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41579,Total threads: 3
Dashboard: http://127.0.0.1:36267/status,Memory: 13.97 GiB
Nanny: tcp://127.0.0.1:39843,


In [3]:
!echo dask dashboard :D
!echo https://jupyterhub.dkrz.de/user/$USER/levante-spawner-preset/proxy/8787/status

dask dashboard :D


https://jupyterhub.dkrz.de/user/b383184/levante-spawner-preset/proxy/8787/status


In [4]:
stores = sorted(Path("/work/bk1450/b383184/Amazon/Atlantic/data/tracks_3456").glob("Parcels_run_*_*.zarr"))
stores[:3]

[PosixPath('/work/bk1450/b383184/Amazon/Atlantic/data/tracks_3456/Parcels_run_3456_2022-06-05T00:00:00.zarr'),
 PosixPath('/work/bk1450/b383184/Amazon/Atlantic/data/tracks_3456/Parcels_run_3456_2022-06-10T00:00:00.zarr'),
 PosixPath('/work/bk1450/b383184/Amazon/Atlantic/data/tracks_3456/Parcels_run_3456_2022-06-15T00:00:00.zarr')]

In [5]:
ds_list = [xr.open_zarr(s) for s in stores]
ds = xr.concat(ds_list,dim='trajectory')

ds

<xarray.Dataset> Size: 36GB
Dimensions:     (trajectory: 1940000, obs: 925)
Coordinates:
  * obs         (obs) int32 4kB 0 1 2 3 4 5 6 7 ... 918 919 920 921 922 923 924
  * trajectory  (trajectory) int64 16MB 0 1 2 3 4 5 ... 9995 9996 9997 9998 9999
Data variables:
    lat         (trajectory, obs) float32 7GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
    lon         (trajectory, obs) float32 7GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 14GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
    z           (trajectory, obs) float32 7GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        JITParticleAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [6]:
num_valid_obs_steps = int(ds.lat.notnull().any('trajectory').sum().compute().data[()])
ds = ds.isel(obs=slice(None, num_valid_obs_steps))
ds

<xarray.Dataset> Size: 29GB
Dimensions:     (trajectory: 1940000, obs: 741)
Coordinates:
  * obs         (obs) int32 3kB 0 1 2 3 4 5 6 7 ... 734 735 736 737 738 739 740
  * trajectory  (trajectory) int64 16MB 0 1 2 3 4 5 ... 9995 9996 9997 9998 9999
Data variables:
    lat         (trajectory, obs) float32 6GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
    lon         (trajectory, obs) float32 6GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 12GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
    z           (trajectory, obs) float32 6GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        JITParticleAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [7]:
ds = ds.assign(start_time=ds.time.isel(obs=0).compute())
ds = ds.assign_coords(start_time=ds.start_time.astype('datetime64[ns]'))
ds


<xarray.Dataset> Size: 29GB
Dimensions:     (trajectory: 1940000, obs: 741)
Coordinates:
  * obs         (obs) int32 3kB 0 1 2 3 4 5 6 7 ... 734 735 736 737 738 739 740
  * trajectory  (trajectory) int64 16MB 0 1 2 3 4 5 ... 9995 9996 9997 9998 9999
    start_time  (trajectory) datetime64[ns] 16MB 2022-06-05 ... 2025-06-05
Data variables:
    lat         (trajectory, obs) float32 6GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
    lon         (trajectory, obs) float32 6GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 12GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
    z           (trajectory, obs) float32 6GB dask.array<chunksize=(10000, 185), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        JITParticleAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [8]:
import numpy as np
np.unique(ds.start_time)

array(['2022-06-05T00:00:00.000000000', '2022-06-10T00:00:00.000000000',
       '2022-06-15T00:00:00.000000000', '2022-06-20T00:00:00.000000000',
       '2022-06-25T00:00:00.000000000', '2022-06-30T00:00:00.000000000',
       '2022-07-05T00:00:00.000000000', '2022-07-10T00:00:00.000000000',
       '2022-07-15T00:00:00.000000000', '2022-07-20T00:00:00.000000000',
       '2022-07-25T00:00:00.000000000', '2022-07-30T00:00:00.000000000',
       '2022-08-04T00:00:00.000000000', '2022-08-09T00:00:00.000000000',
       '2022-08-14T00:00:00.000000000', '2022-08-19T00:00:00.000000000',
       '2022-08-24T00:00:00.000000000', '2022-08-29T00:00:00.000000000',
       '2022-09-03T00:00:00.000000000', '2022-09-08T00:00:00.000000000',
       '2022-09-13T00:00:00.000000000', '2022-09-18T00:00:00.000000000',
       '2022-09-23T00:00:00.000000000', '2022-09-28T00:00:00.000000000',
       '2022-10-03T00:00:00.000000000', '2022-10-08T00:00:00.000000000',
       '2022-10-13T00:00:00.000000000', '2022-10-18

In [9]:
# pick the day you want
import numpy as np
import matplotlib.pyplot as plt

### Animations

In [10]:
import os
import numpy as np
import xarray as xr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import PillowWriter
from matplotlib.collections import LineCollection
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def animate_cartopy_multidate(ds, start_dates, out_gif,
                              trail_len=12, fps=8, dpi=100,
                              coast_res="110m"):
    """
    Animate multiple start dates in the same video with different colors
    
    - ds: Parcels zarr xarray.Dataset
    - start_dates: list of np.datetime64 dates, e.g., [np.datetime64('2022-06-05'), ...]
    - out_gif: output path
    """
    print(f"Starting multi-date GIF animation for {len(start_dates)} dates")
    
    # Define colors for each date (cycle through if more dates than colors)
    colors = ['navy', 'darkred', 'darkgreen', 'purple', 'darkorange', 'brown', 'pink', 'gray']
    tail_colors = ['deepskyblue', 'lightcoral', 'lightgreen', 'plum', 'orange', 'tan', 'lightpink', 'lightgray']
    
    # 1) Filter and combine data for all dates
    all_data = []
    all_finite = []
    date_labels = []
    
    for i, target_date in enumerate(start_dates):
        print(f"Processing date {i+1}/{len(start_dates)}: {target_date}")
        
        mask = (ds.start_time.dt.floor("D") == target_date)
        ds_day = ds.isel(trajectory=mask)
        n_selected = int(mask.sum().compute())
        
        if n_selected == 0:
            print(f"  No trajectories for {target_date}, skipping")
            continue
            
        print(f"  Selected {n_selected} trajectories")
        
        lon = ds_day.lon.compute().astype("float32").values
        lat = ds_day.lat.compute().astype("float32").values
        
        # Store data for this date
        all_data.append({
            'lon': lon,
            'lat': lat,
            'color': colors[i % len(colors)],
            'tail_color': tail_colors[i % len(tail_colors)],
            'date': target_date,
            'n_traj': n_selected
        })
        
        finite = np.isfinite(lon) & np.isfinite(lat)
        all_finite.append(finite)
        date_labels.append(np.datetime_as_string(target_date, 'D'))
    
    if not all_data:
        raise ValueError("No valid data found for any of the specified dates")
    
    # 2) Get frame times (assume all dates have same time axis)
    t_da = ds.time
    if "trajectory" in t_da.dims:
        frame_times = t_da.isel(trajectory=0).compute().values
    else:
        frame_times = t_da.compute().values
    
    # 3) Find frames that have data for any date
    combined_finite = np.any([finite.any(axis=0) for finite in all_finite], axis=0)
    frame_idx = np.flatnonzero(combined_finite)
    
    # Subsample frames
    frame_idx = frame_idx[::3]
    print(f"Using {len(frame_idx)} frames (subsampled)")
    
    # 4) Map extents (from all dates combined)
    all_lons = np.concatenate([data['lon'].flatten() for data in all_data])
    all_lats = np.concatenate([data['lat'].flatten() for data in all_data])
    
    xmin = np.nanmin(all_lons); xmax = np.nanmax(all_lons)
    ymin = np.nanmin(all_lats); ymax = np.nanmax(all_lats)
    dx = xmax - xmin; dy = ymax - ymin
    padx, pady = 0.05*dx, 0.05*dy
    
    print(f"Map bounds: lon=[{xmin:.2f}, {xmax:.2f}], lat=[{ymin:.2f}, {ymax:.2f}]")
    
    # 5) Setup figure
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(10, 8), dpi=dpi)  # Bigger for multiple datasets
    ax = plt.axes(projection=proj)
    ax.set_extent([xmin - padx, xmax + padx, ymin - pady, ymax + pady], crs=proj)
    
    ax.add_feature(cfeature.LAND.with_scale(coast_res), facecolor="#f2f2f2", edgecolor="none", zorder=0)
    ax.coastlines(resolution=coast_res, linewidth=0.5, zorder=1)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    
    # 6) Create artists for each date
    artists = []
    L = max(2, int(trail_len))
    
    for i, data in enumerate(all_data):
        N = data['lon'].shape[0]
        max_segments = N * (L - 1)
        segs = np.empty((max_segments, 2, 2), dtype=np.float32)
        
        # Tail for this date
        tail = LineCollection([], colors=data['tail_color'], linewidths=1.2, alpha=0.7,
                              transform=proj, zorder=2+i)
        ax.add_collection(tail)
        
        # Head for this date
        head_offsets = np.empty((N, 2), dtype=np.float32)
        head = ax.scatter([], [], s=16, c=data['color'], marker="o", linewidths=0, 
                          edgecolors="none", transform=proj, zorder=10+i)
        
        artists.append({
            'tail': tail,
            'head': head,
            'segs': segs,
            'head_offsets': head_offsets,
            'data': data
        })
    
    # Legend
    legend_elements = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=data['color'], 
                                  markersize=8, label=f"{np.datetime_as_string(data['date'], 'D')} (n={data['n_traj']})")
                       for data in all_data]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=8)
    
    title = ax.set_title("")
    
    # 7) Animation loop
    try:
        print("Starting GIF creation...")
        writer = PillowWriter(fps=fps)
        
        with writer.saving(fig, out_gif, dpi=dpi):
            for frame_i, k in enumerate(frame_idx):
                if frame_i % 20 == 0:
                    print(f"Frame {frame_i+1}/{len(frame_idx)}")
                
                total_particles = 0
                
                # Update each date's particles
                for artist in artists:
                    data = artist['data']
                    lon = data['lon']
                    lat = data['lat']
                    finite = np.isfinite(lon) & np.isfinite(lat)
                    
                    # Tail window
                    s = max(0, k - (L - 1))
                    
                    if s == 0 and k < L - 1:
                        N = lon.shape[0]
                        xw = np.full((N, L), np.nan, dtype=np.float32)
                        yw = np.full((N, L), np.nan, dtype=np.float32)
                        actual_len = k + 1
                        xw[:, -actual_len:] = lon[:, :k+1]
                        yw[:, -actual_len:] = lat[:, :k+1]
                    else:
                        xw = lon[:, s:k+1]
                        yw = lat[:, s:k+1]
                        if xw.shape[1] < L:
                            N = lon.shape[0]
                            pad_len = L - xw.shape[1]
                            xw_pad = np.full((N, pad_len), np.nan, dtype=np.float32)
                            yw_pad = np.full((N, pad_len), np.nan, dtype=np.float32)
                            xw = np.concatenate([xw_pad, xw], axis=1)
                            yw = np.concatenate([yw_pad, yw], axis=1)
                    
                    # Build segments
                    if xw.shape[1] >= 2:
                        x0 = xw[:, :-1]; y0 = yw[:, :-1]
                        x1 = xw[:,  1:]; y1 = yw[:,  1:]
                        mm = np.isfinite(x0) & np.isfinite(y0) & np.isfinite(x1) & np.isfinite(y1)
                        
                        if mm.any():
                            xv0 = x0[mm]; yv0 = y0[mm]
                            xv1 = x1[mm]; yv1 = y1[mm]
                            nseg = xv0.size
                            artist['segs'][:nseg, 0, 0] = xv0
                            artist['segs'][:nseg, 0, 1] = yv0
                            artist['segs'][:nseg, 1, 0] = xv1
                            artist['segs'][:nseg, 1, 1] = yv1
                            artist['tail'].set_segments(artist['segs'][:nseg])
                        else:
                            artist['tail'].set_segments([])
                    else:
                        artist['tail'].set_segments([])
                    
                    # Heads
                    m = finite[:, k]
                    nheads = int(m.sum())
                    total_particles += nheads
                    
                    if nheads > 0:
                        artist['head_offsets'][:nheads, 0] = lon[m, k]
                        artist['head_offsets'][:nheads, 1] = lat[m, k]
                        artist['head'].set_offsets(artist['head_offsets'][:nheads])
                    else:
                        artist['head'].set_offsets(np.empty((0, 2), dtype=np.float32))
                
                # Title
                try:
                    ts = np.datetime_as_string(frame_times[k], unit="D")
                    title.set_text(f"{ts}  (total particles: {total_particles})")
                except:
                    title.set_text(f"Frame {k}  (total particles: {total_particles})")
                
                writer.grab_frame()
        
        plt.close(fig)
        print(f"Multi-date GIF complete! Saved to: {out_gif}")
        return out_gif
        
    except Exception as e:
        plt.close(fig)
        print(f"Error during GIF creation: {e}")
        raise



In [11]:
# Usage with multiple dates:
start_dates = [
    np.datetime64('2022-10-28'),
    np.datetime64('2022-11-07'),
    np.datetime64('2022-11-17'),
    np.datetime64('2022-11-27'),
    np.datetime64('2022-12-07'),
    np.datetime64('2022-12-17'),
    np.datetime64('2022-12-27'),
    np.datetime64('2023-01-06'),
    np.datetime64('2023-01-16'),
    np.datetime64('2023-01-26'),
    np.datetime64('2023-02-05'),
    np.datetime64('2023-02-15'),
    np.datetime64('2023-02-25'),
    np.datetime64('2023-03-07'),
]

out_dir = "/work/bk1450/b383184/Amazon/Atlantic/animations"
os.makedirs(out_dir, exist_ok=True)
out_gif = os.path.join(out_dir, "trajectories_multidate_2.gif")

path = animate_cartopy_multidate(ds, start_dates, out_gif, trail_len=40, fps=6, dpi=100, coast_res="110m")
print("SUCCESS: Multi-date animation saved to:", path)

Starting multi-date GIF animation for 14 dates
Processing date 1/14: 2022-10-28
  Selected 10000 trajectories


Processing date 2/14: 2022-11-07
  Selected 10000 trajectories


Processing date 3/14: 2022-11-17
  Selected 10000 trajectories


Processing date 4/14: 2022-11-27
  Selected 10000 trajectories


Processing date 5/14: 2022-12-07
  Selected 10000 trajectories


...

Frame 181/247


Frame 201/247


Frame 221/247


Frame 241/247


Multi-date GIF complete! Saved to: /work/bk1450/b383184/Amazon/Atlantic/animations/trajectories_multidate_2.gif
SUCCESS: Multi-date animation saved to: /work/bk1450/b383184/Amazon/Atlantic/animations/trajectories_multidate_2.gif
